# PaddleOCR handwriting extraction

Install `requirements.txt` in your project environment, then select its notebook kernel (for example, `omscs`). Use `RUN_MODE='run'` for the first run and `RUN_MODE='review'` to inspect saved results. Configuration is in the setup cell below.

- `RUN_MODE='review'`: display saved predictions and field crops.
- `RUN_MODE='run'`: run PaddleOCR on CPU and replace the saved prediction JSON/CSV pair after successful recognition.

The final cell displays crops, raw OCR, and values parsed using `template/ocr_rules.json`.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_DIRECTORY = None  # Auto-detect; set an absolute path if needed.
print('Kernel Python:', sys.executable)
PROJECT = Path(PROJECT_DIRECTORY).expanduser().resolve() if PROJECT_DIRECTORY else None
for base in (() if PROJECT is not None else (Path.cwd(), *Path.cwd().parents)):
    for candidate in (base, base / 'id-card-parsing', base / 'Lizard_Toepads/id-card-parsing'):
        if (candidate / 'card_ocr/notebook.py').is_file():
            PROJECT = candidate.resolve()
            break
    if PROJECT is not None:
        break
if PROJECT is None or not (PROJECT / 'card_ocr/notebook.py').is_file():
    raise FileNotFoundError(
        'The kernel cannot access id-card-parsing. A local notebook connected to a Colab server '
        'executes remotely and cannot read your Mac files. Select the local project kernel, '
        'or copy the project, data, and template to the remote kernel and set PROJECT_DIRECTORY there. '
        'For a local kernel, start Jupyter inside the project or set PROJECT_DIRECTORY explicitly.'
    )
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from card_ocr.notebook import (
    choose_run, command, latest_predictions,
    run_paddle,
)
from card_ocr.images import select_images
from card_ocr.fields import load_layout, load_checkboxes
from card_ocr.recipe import preparation_recipe
from card_ocr.extraction import extract_records, load_rules

PYTHON = Path(sys.executable)
RULES_PATH = PROJECT / 'template/ocr_rules.json'
load_rules(RULES_PATH)
RUN_MODE = 'review'
START_NEW_RUN = False
CLEAN_PRINT = True
if RUN_MODE not in ('review', 'run'):
    raise ValueError("Choose RUN_MODE='review' or 'run'.")
SAMPLE_COUNT = 5
IMAGE_IDS = ['0010', '0011', '0014', '0015', '0043']  # None selects the first SAMPLE_COUNT images.
FIELDS = ['day', 'month', 'year', 'time', 'flag', 'tb', 'ta', 'ph', 'pw',
          'perch', 'recapture_id', 'svl', 'mass', 'initials']
IDS = select_images(PROJECT / 'data', ids=IMAGE_IDS, count=SAMPLE_COUNT)
layout = [
    field for field in load_layout(PROJECT / 'template/fields.json')
    if field.name in FIELDS
]
if set(FIELDS) != {field.name for field in layout}:
    raise ValueError('FIELDS contains an unknown field name.')
RECIPE = preparation_recipe(
    PROJECT / 'template/template.jpg', layout, clean_print=CLEAN_PRINT,
    checkboxes=load_checkboxes(PROJECT / 'template/fields.json'),
)
RUN = choose_run(
    PROJECT, IDS, FIELDS, start_new=START_NEW_RUN,
    review=RUN_MODE == 'review', recipe=RECIPE,
)
print('Project:', PROJECT)
print('OCR Python:', PYTHON)
print('Images:', len(IDS), '| fields per image:', len(FIELDS))
print('Run:', RUN)
print('Mode:', RUN_MODE)


## Prepare field crops

Matching crops are reused. In run mode, missing crops are prepared automatically; `START_NEW_RUN=True` rebuilds `results/ocr/current/` after successful preparation, clearing predictions for the previous crops. Review mode requires saved predictions.

`IMAGE_IDS` selects cards; `None` uses the first `SAMPLE_COUNT` images. `FIELDS` selects regions from `template/fields.json`. `CLEAN_PRINT=True` masks template printing while retaining raw crops.

Preparation saves aligned raw crops, cleaned OCR inputs, and checkbox measurements in the manifest.


In [ ]:
if (RUN / 'manifest.json').is_file() and not (RUN_MODE == 'run' and START_NEW_RUN):
    print('Reusing prepared crops:', RUN)
else:
    command(
        PROJECT, PYTHON, 'prepare', '--ids', *IDS, '--fields', *FIELDS,
        '--output', RUN,
        *(['--replace'] if START_NEW_RUN else []),
        *([] if CLEAN_PRINT else ['--keep-print']),
    )
manifest = json.loads((RUN / 'manifest.json').read_text())
print('Prepared fields:', len(manifest['samples']))
if manifest['errors']:
    print('Images needing review:', manifest['errors'])


In [ ]:
from IPython.display import display, Image as NotebookImage

display(NotebookImage(filename=str(RUN / 'template_fields.png')))
for image_id in IDS[:3]:
    overlay = RUN / image_id / 'fields.png'
    if overlay.exists():
        print('Field regions:', image_id)
        display(NotebookImage(filename=str(overlay)))
print('All scan/card overlays and individual field crops are in:', RUN)

print('Checkbox suggestions (not OCR text; confirm against checkboxes.png):')
for image_id in IDS:
    boxes = [item for item in manifest.get('checkboxes', []) if item['image_id'] == image_id]
    print(image_id, {item['field']: item['state'] for item in boxes})


## Run PaddleOCR

`RUN_MODE='run'` starts the CPU recognizer in a separate process. Successful recognition replaces the saved prediction JSON/CSV pair. The checkpoint is downloaded to the environment cache on first use. Interrupting the cell stops recognition.


In [ ]:
if RUN_MODE == 'run':
    PREDICTIONS = run_paddle(PROJECT, PYTHON, RUN, rules_path=RULES_PATH)
    print('JSON:', PREDICTIONS)
    print('CSV:', PREDICTIONS.with_suffix('.csv'))
else:
    print('Paddle prediction skipped in review mode.')


In [ ]:
saved = latest_predictions(RUN)
if 'paddle' in saved:
    payload = json.loads(saved['paddle'].read_text())
    failures = sum(row['status'] != 'ok' for row in payload['predictions'])
    print('Paddle checkpoint:', payload.get('model_id'))
    print('Saved fields:', len(payload['predictions']), '| failed:', failures)
else:
    print('No saved Paddle predictions. Set RUN_MODE="run" to generate them.')


## Review predictions

Each field displays its crop, unchanged OCR text, and extracted value. Current JSON rules format times and numbers and combine day/month/year into a date. Ambiguous or invalid values stay empty; no mismatch flags are shown.

`SHOW_IMAGE_IDS` selects displayed cards. `SHOW_RAW_CROPS=True` also displays uncleaned crops.


In [ ]:
from importlib import reload
import card_ocr.extraction as extraction

extraction = reload(extraction)
from IPython.display import display, Image as NotebookImage

SHOW_IMAGE_IDS = IDS
SHOW_RAW_CROPS = False
comparison_paths = latest_predictions(RUN)
prediction_path = comparison_paths.get('paddle')
raw_rows = json.loads(prediction_path.read_text())['predictions'] if prediction_path else []
comparison_rows = {(row['image_id'], row['field']): row for row in raw_rows}
extracted = {row['image_id']: row for row in extraction.extract_records(raw_rows, extraction.load_rules(RULES_PATH))}
comparison_manifest = json.loads((RUN / 'manifest.json').read_text())
for image_id in SHOW_IMAGE_IDS:
    print('Card', image_id, '| date:', extracted.get(image_id, {}).get('date'))
    samples = [row for row in comparison_manifest['samples'] if row['image_id'] == image_id]
    if not samples:
        print('No prepared crops for image:', image_id)
    for sample in samples:
        key = (sample['image_id'], sample['field'])
        print('\nImage', key[0], '| field:', key[1])
        if SHOW_RAW_CROPS and sample.get('raw_crop'):
            print('Raw aligned crop:')
            display(NotebookImage(filename=str(RUN / sample['raw_crop'])))
        print('OCR input:')
        display(NotebookImage(filename=str(RUN / sample['crop'])))
        prediction = comparison_rows.get(key)
        if prediction_path is None:
            print('paddle: not run')
        elif prediction is None:
            print('paddle: no prediction for this field')
        else:
            print(f"Raw OCR: {prediction['text']!r}")
            if prediction.get('retry_text') is not None:
                print('Retry OCR:', repr(prediction['retry_text']))
            print('Extracted:', extracted.get(image_id, {}).get(key[1]))
            if prediction.get('error'):
                print('  Error:', prediction['error'])
